<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Taylor_Series_Function_Approximation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Project Overview: Taylor Series Visualizations

This notebook generates high-quality, vertical (9:16) animations designed for social media. The primary focus is the visualization of Taylor Series approximations for various fundamental mathematical functions. By building these approximations term-by-term, the animation demonstrates how complex curves can be reconstructed using simple polynomials.

### Mathematical Background

A Taylor series is a representation of a function as an infinite sum of terms that are calculated from the values of the function's derivatives at a single point. When that point is zero, the series is specifically referred to as a Maclaurin series.

The general formula for a Taylor series of a real or complex-valued function $f(x)$ that is infinitely differentiable at a real or complex number $a$ is the power series:

$$f(x) = f(a) + \frac{f'(a)}{1!}(x-a) + \frac{f''(a)}{2!}(x-a)^2 + \frac{f'''(a)}{3!}(x-a)^3 + \dots$$

In this notebook, we focus on Maclaurin series ($a=0$) for:
1. **Sine ($ \sin x $)**: Showcasing how odd functions only utilize odd powers of $x$.
2. **Cosine ($ \cos x $)**: Demonstrating how even functions only utilize even powers of $x$.
3. **Exponential ($ e^x $)**: Illustrating a series that converges for all real numbers.
4. **Geometric ($ 1/(1-x) $)**: Highlighting the concept of the radius of convergence, where the approximation is only valid for $|x| < 1$.

### Implementation Details

- **Rendering**: Uses Matplotlib's `FuncAnimation` to generate MP4 files.
- **Formatting**: Designed for 1080x1920 resolution (portrait).
- **Control**: Parameters like `MAX_TERMS` and `FPS` allow for easy adjustment of the animation's complexity and smoothness.
- **Output**: The script automatically handles bitrate calculation to keep the file size optimized for web sharing.

In [ ]:
# 1. Install system dependencies and Manim
!sudo apt-get update
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg dvisvgm texlive-full
!pip install manim

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: dpkg was interrupted, you must manually run 'sudo dpkg --configure -a' to correct the problem. 


In [ ]:
%load_ext manim

The manim module is not an IPython extension.


In [1]:
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Concept: Taylor Series - building a function's approximation one term at a time
Repository: github.com/zombimann/Mathematical-video-animations-and-visualization

A vertical (9:16) social-media animation. For a sequence of diverse functions,
the exact curve is plotted and its Maclaurin (Taylor about a = 0) polynomial is
grown term-by-term, from one term up to six, smoothly morphing onto the true
curve while the live polynomial expression is rendered in a bottom card.
Everything is parametric for quick experimentation.
"""

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import FancyBboxPatch, Rectangle

# --------------------------------------------------------------------------------------
# 1. PARAMETERS  (tweak freely)
# --------------------------------------------------------------------------------------
OUTPUT        = "taylor_series_short.mp4"
FPS           = 30
DPI           = 200                 # 5.4 x 9.6 in @ 200 dpi -> 1080 x 1920 (9:16)
FIG_W, FIG_H  = 5.4, 9.6
MAX_TERMS     = 6                   # grow up to this many terms per function
TARGET_MB     = 8.5                 # keep file comfortably < 10 MB
AUTO_DOWNLOAD = True

# Beat durations (seconds)
INTRO_DUR, ESTABLISH_DUR, TERM_DUR, SETTLE_DUR, OUTRO_DUR = 3.0, 1.2, 1.2, 1.2, 1.8
MORPH_FRAC = 0.60                   # fraction of a term-beat spent morphing the curve
WRAP       = 3                      # max polynomial terms per rendered line

# Palette
BG    = "#0B1020"   # deep navy background
CARD  = "#121A2E"   # card panel
INK   = "#E8EDF7"   # primary text
MUTED = "#8A94A6"   # secondary text
GRID  = "#1C2740"   # grid lines
AXIS  = "#2A3550"   # zero axes
TRUE  = "#C7D0E0"   # exact-curve colour
BRAND = "#3D5AFE"   # intro accents + closing-card background

plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams["font.family"]      = "DejaVu Sans"

# --------------------------------------------------------------------------------------
# 2. THE FUNCTIONS  (subject-matter named; each term carries its own sign)
# --------------------------------------------------------------------------------------
def const(c):
    return lambda x: np.full_like(x, float(c))

SCENES = [
    dict(
        name="SINE", color="#38BDF8", eq_fs=24,
        fx=r"$f(x)=\sin x$",
        f=np.sin,
        xlim=(-5, 5), ylim=(-2.8, 2.8),
        note="Odd function: only odd powers of x appear",
        terms=[lambda x: x,
               lambda x: -x**3/6,
               lambda x:  x**5/120,
               lambda x: -x**7/5040,
               lambda x:  x**9/362880,
               lambda x: -x**11/39916800],
        tex=[r"x", r"-\frac{x^{3}}{3!}", r"+\frac{x^{5}}{5!}",
             r"-\frac{x^{7}}{7!}", r"+\frac{x^{9}}{9!}", r"-\frac{x^{11}}{11!}"],
    ),
    dict(
        name="COSINE", color="#F472B6", eq_fs=24,
        fx=r"$f(x)=\cos x$",
        f=np.cos,
        xlim=(-5, 5), ylim=(-2.8, 2.8),
        note="Even function: only even powers of x appear",
        terms=[const(1),
               lambda x: -x**2/2,
               lambda x:  x**4/24,
               lambda x: -x**6/720,
               lambda x:  x**8/40320,
               lambda x: -x**10/3628800],
        tex=[r"1", r"-\frac{x^{2}}{2!}", r"+\frac{x^{4}}{4!}",
             r"-\frac{x^{6}}{6!}", r"+\frac{x^{8}}{8!}", r"-\frac{x^{10}}{10!}"],
    ),
    dict(
        name="EXPONENTIAL", color="#FBBF24", eq_fs=26,
        fx=r"$f(x)=e^{x}$",
        f=np.exp,
        xlim=(-3, 2), ylim=(-1.5, 8),
        note="Converges for every real x",
        terms=[const(1),
               lambda x: x,
               lambda x: x**2/2,
               lambda x: x**3/6,
               lambda x: x**4/24,
               lambda x: x**5/120],
        tex=[r"1", r"+x", r"+\frac{x^{2}}{2!}",
             r"+\frac{x^{3}}{3!}", r"+\frac{x^{4}}{4!}", r"+\frac{x^{5}}{5!}"],
    ),
    dict(
        name="GEOMETRIC", color="#34D399", eq_fs=28,
        fx=r"$f(x)=1/(1-x)$",
        f=lambda x: 1.0/(1.0 - x),
        xlim=(-0.85, 0.85), ylim=(-1.5, 7),
        note="Converges only for |x| < 1",
        terms=[const(1),
               lambda x: x,
               lambda x: x**2,
               lambda x: x**3,
               lambda x: x**4,
               lambda x: x**5],
        tex=[r"1", r"+x", r"+x^{2}", r"+x^{3}", r"+x^{4}", r"+x^{5}"],
    ),
]

# --------------------------------------------------------------------------------------
# 3. TIMELINE
# --------------------------------------------------------------------------------------
def smoothstep(t):
    t = min(1.0, max(0.0, t))
    return t * t * (3 - 2 * t)

def clamp(t):
    return min(1.0, max(0.0, t))

beats = []
def add(kind, dur, **kw):
    f0 = beats[-1]["f1"] if beats else 0
    beats.append(dict(kind=kind, f0=f0, f1=f0 + max(1, round(dur * FPS)), **kw))

add("intro", INTRO_DUR)
for si in range(len(SCENES)):
    add("establish", ESTABLISH_DUR, si=si)
    for n in range(1, MAX_TERMS + 1):
        add("term", TERM_DUR, si=si, n=n)
    add("settle", SETTLE_DUR, si=si)
add("outro", OUTRO_DUR)

TOTAL    = beats[-1]["f1"]
DURATION = TOTAL / FPS

def locate(i):
    for b in beats:
        if b["f0"] <= i < b["f1"]:
            return b, (i - b["f0"]) / (b["f1"] - b["f0"])
    return beats[-1], 1.0

# --------------------------------------------------------------------------------------
# 4. DRAWING HELPERS
# --------------------------------------------------------------------------------------
X_PTS = 700

def taylor_y(scene, n, morph, x):
    """Partial sum of the first n terms, newest term scaled by `morph`."""
    y = np.zeros_like(x)
    for k in range(n - 1):
        y += scene["terms"][k](x)
    if n >= 1:
        y += morph * scene["terms"][n - 1](x)
    return y

def poly_tex(scene, n):
    """Live polynomial as 1-2 centred mathtext lines, anchored by T_n(x)=."""
    parts = scene["tex"][:n]
    lines = ["".join(parts[i:i + WRAP]) for i in range(0, len(parts), WRAP)]
    out = r"$T_{%d}(x)=%s$" % (n, lines[0])
    for extra in lines[1:]:
        out += "\n" + r"$%s$" % extra
    return out

def fill_bg(color):
    # Paint the whole frame so the background is baked into the rendered content,
    # independent of how the movie writer treats the figure facecolor.
    fig.add_artist(Rectangle((0, 0), 1, 1, transform=fig.transFigure,
                             facecolor=color, edgecolor="none", zorder=-100))

def watermark():
    fig.text(0.035, 0.022, "\u00A9 Mugambi Ndwiga  /  @craftsandengineering",
             color=INK, alpha=0.60, fontsize=14, ha="left", va="bottom")

def progress_bar(frac, accent):
    fig.add_artist(Rectangle((0, 0.992), 1, 0.008, transform=fig.transFigure,
                             color=GRID, zorder=0))
    fig.add_artist(Rectangle((0, 0.992), clamp(frac), 0.008, transform=fig.transFigure,
                             color=accent, zorder=1))

def draw_card(scene, n, kind, morph, A):
    accent = scene["color"]
    fig.add_artist(FancyBboxPatch(
        (0.06, 0.075), 0.88, 0.195, transform=fig.transFigure,
        boxstyle="round,pad=0.0,rounding_size=0.02",
        facecolor=CARD, edgecolor=accent, linewidth=1.6, alpha=0.92 * A, zorder=2))

    label = "TAYLOR POLYNOMIAL  \u00B7  a = 0" if n == 0 else \
            "TAYLOR POLYNOMIAL  \u00B7  a = 0  \u00B7  n = %d" % n
    fig.text(0.5, 0.250, label, color=MUTED, fontsize=15, ha="center",
             va="center", alpha=A, zorder=3)

    if kind == "term":
        if n >= 2:
            fig.text(0.5, 0.170, poly_tex(scene, n - 1), color=INK,
                     fontsize=scene["eq_fs"], ha="center", va="center",
                     alpha=A * clamp(1 - morph / 0.30), zorder=3)
        fig.text(0.5, 0.170, poly_tex(scene, n), color=INK,
                 fontsize=scene["eq_fs"], ha="center", va="center",
                 alpha=A * clamp(morph / 0.45), zorder=3)
    elif kind == "settle":
        fig.text(0.5, 0.170, poly_tex(scene, MAX_TERMS), color=INK,
                 fontsize=scene["eq_fs"], ha="center", va="center",
                 alpha=A, zorder=3)
    # establish (n == 0): empty labelled card, approximation not started yet

def draw_function(scene, n, morph, A, kind):
    accent = scene["color"]

    ax = fig.add_axes([0.135, 0.345, 0.775, 0.470])
    ax.set_facecolor("none")
    ax.set_xlim(*scene["xlim"]); ax.set_ylim(*scene["ylim"])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.axhline(0, color=AXIS, lw=1.2, alpha=A)
    ax.axvline(0, color=AXIS, lw=1.2, alpha=A)
    ax.grid(True, color=GRID, lw=0.8, alpha=0.45 * A)
    ax.tick_params(colors=MUTED, labelsize=13, length=0)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_alpha(A)

    x = np.linspace(scene["xlim"][0], scene["xlim"][1], X_PTS)
    ax.plot(x, scene["f"](x), color=TRUE, lw=5.0, alpha=0.50 * A, solid_capstyle="round")

    if n >= 1:
        yt = taylor_y(scene, n, morph, x)
        ax.plot(x, yt, color=accent, lw=3.6, alpha=A, solid_capstyle="round")
        if kind == "settle":   # shade the shrinking residual error
            ax.fill_between(x, scene["f"](x), yt, color=accent, alpha=0.12 * A, lw=0)

    name_fs = min(44, int(380 / max(1, len(scene["name"]))))
    fig.text(0.5, 0.955, scene["name"], color=accent, fontsize=name_fs,
             fontweight="bold", ha="center", va="center", alpha=A)
    fig.text(0.5, 0.905, scene["fx"], color=INK, fontsize=30,
             ha="center", va="center", alpha=A)
    fig.text(0.30, 0.857, r"$\mathdefault{--}$ f(x) exact", color=TRUE,
             fontsize=16, ha="center", va="center", alpha=A)
    fig.text(0.66, 0.857, r"$\mathdefault{--}$ $T_n(x)$ approx", color=accent,
             fontsize=16, ha="center", va="center", alpha=A)
    fig.text(0.5, 0.305, scene["note"], color=MUTED, fontsize=15,
             ha="center", va="center", alpha=A)

    draw_card(scene, n, kind, morph, A)

def draw_intro(local):
    a = smoothstep(clamp(local / 0.30)) * (1 - smoothstep(clamp((local - 0.80) / 0.20)))
    fig.text(0.5, 0.705, "TAYLOR", color=INK, fontsize=50,
             fontweight="bold", ha="center", va="center", alpha=a)
    fig.text(0.5, 0.633, "SERIES", color=INK, fontsize=50,
             fontweight="bold", ha="center", va="center", alpha=a)
    fig.text(0.5, 0.560, "Approximating functions",
             color=BRAND, fontsize=20, ha="center", va="center", alpha=a)
    fig.text(0.5, 0.520, "one term at a time",
             color=BRAND, fontsize=20, ha="center", va="center", alpha=a)
    fig.text(0.5, 0.425,
             r"$f(x)=\sum_{n=0}^{\infty}\frac{f^{(n)}(a)}{n!}\,(x-a)^{n}$",
             color=INK, fontsize=28, ha="center", va="center", alpha=a)
    fig.text(0.5, 0.345, "Here a = 0   (Maclaurin series)",
             color=MUTED, fontsize=17, ha="center", va="center", alpha=a)
    fig.text(0.5, 0.298, r"$f(x)$ exact     vs     $T_n(x)$ approximation",
             color=MUTED, fontsize=17, ha="center", va="center", alpha=a)

def draw_outro(local):
    a = smoothstep(clamp(local / 0.25))
    fig.text(0.5, 0.560, "Made by", color="white", fontsize=24,
             ha="center", va="center", alpha=a)
    fig.text(0.5, 0.495, "Mugambi Ndwiga", color="white", fontsize=33,
             fontweight="bold", ha="center", va="center", alpha=a)
    fig.text(0.5, 0.430, "@craftsandengineering", color="white", fontsize=26,
             ha="center", va="center", alpha=a)

# --------------------------------------------------------------------------------------
# 5. FRAME RENDERER
# --------------------------------------------------------------------------------------
fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=DPI)

def render(i):
    fig.clf()
    b, local = locate(i)

    if b["kind"] == "outro":
        fill_bg(BRAND); draw_outro(local); return

    fill_bg(BG)

    if b["kind"] == "intro":
        draw_intro(local); progress_bar(i / TOTAL, BRAND); watermark(); return

    scene = SCENES[b["si"]]
    if b["kind"] == "establish":
        A, n, morph = smoothstep(local), 0, 0.0
    elif b["kind"] == "term":
        A, n = 1.0, b["n"]
        morph = smoothstep(clamp(local / MORPH_FRAC))
    else:  # settle
        n, morph = MAX_TERMS, 1.0
        A = 1.0 - smoothstep(clamp((local - 0.55) / 0.45))

    draw_function(scene, n, morph, A, b["kind"])
    progress_bar(i / TOTAL, scene["color"]); watermark()

# --------------------------------------------------------------------------------------
# 6. RENDER + ENCODE  (bitrate auto-tuned to stay < 10 MB)
# --------------------------------------------------------------------------------------
bitrate = int(TARGET_MB * 8 * 1024 / DURATION * 0.92)   # kbps
print("Frames: %d   Duration: %.1fs   Target bitrate: %d kbps" % (TOTAL, DURATION, bitrate))

anim = animation.FuncAnimation(fig, render, frames=TOTAL, interval=1000 / FPS, blit=False)
writer = animation.FFMpegWriter(
    fps=FPS, bitrate=bitrate, codec="libx264",
    extra_args=["-pix_fmt", "yuv420p", "-movflags", "+faststart"])
anim.save(OUTPUT, writer=writer, dpi=DPI)
plt.close(fig)

size_mb = os.path.getsize(OUTPUT) / 1e6
print("Saved %s  (%.2f MB)" % (OUTPUT, size_mb))

# --------------------------------------------------------------------------------------
# 7. DISPLAY + DOWNLOAD  (least-problematic path across environments)
# --------------------------------------------------------------------------------------
from IPython.display import Video, display
display(Video(OUTPUT, embed=True, width=360))

if AUTO_DOWNLOAD:
    try:
        from google.colab import files
        files.download(OUTPUT)
    except Exception:
        try:
            from IPython.display import FileLink
            display(FileLink(OUTPUT))
        except Exception:
            print("Download manually from:", os.path.abspath(OUTPUT))

Frames: 1296   Duration: 43.2s   Target bitrate: 1482 kbps
Saved taylor_series_short.mp4  (6.98 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>